In [ ]:
# Machine Learning on Conductivity Fluctuation Features

This notebook uses spectral features extracted from conductivity fluctuation signals to analyze and classify thin metal films.

Models:
- K-Means clustering
- K-Nearest Neighbors
- Fully connected neural network

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    silhouette_score,
)

In [ ]:
DATASET_PATH = Path("../data/processed/conductivity_features.csv")
RESULTS_DIR = Path("../results")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATASET_PATH)

df.head()

In [ ]:
MATERIAL_ORDER = ["BI", "CR", "FE", "XXX"]

In [ ]:
df.info()
df["material"].value_counts()

In [ ]:
FEATURES = ["gamma", "energy"]

X = df[FEATURES].copy()
y = df["material"].copy()

# Energy can vary by several orders of magnitude
X["energy"] = np.log10(X["energy"].clip(lower=1e-30))

X.describe()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
n_clusters = y.nunique()

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=20,
)

cluster_labels = kmeans.fit_predict(X_train_scaled)

silhouette = silhouette_score(X_train_scaled, cluster_labels)

print(f"Number of clusters: {n_clusters}")
print(f"Silhouette score: {silhouette:.4f}")

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    x=X_train_scaled[:, 0],
    y=X_train_scaled[:, 1],
    hue=cluster_labels,
    palette="tab10",
    s=25,
)

plt.xlabel("Gamma (scaled)")
plt.ylabel("Log10 energy (scaled)")
plt.title("K-Means clustering of conductivity fluctuation features")

plt.tight_layout()
plt.savefig("../results/kmeans_clusters.png", dpi=300)
plt.show()

In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
)

knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_test_scaled)

knn_accuracy = accuracy_score(y_test, knn_pred)

print(f"KNN accuracy: {knn_accuracy * 100:.2f}%")
print(
    classification_report(
        y_test,
        knn_pred,
        labels=MATERIAL_ORDER,
        target_names=MATERIAL_ORDER,
        zero_division=0,
    )
)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    max_iter=1000,
    early_stopping=True,
    random_state=42,
)

mlp.fit(X_train_scaled, y_train)

mlp_pred = mlp.predict(X_test_scaled)

mlp_accuracy = accuracy_score(y_test, mlp_pred)

print(f"Neural network accuracy: {mlp_accuracy * 100:.2f}%")
print(
    classification_report(
        y_test,
        mlp_pred,
        labels=MATERIAL_ORDER,
        target_names=MATERIAL_ORDER,
        zero_division=0,
    )
)

In [ ]:
cm = confusion_matrix(y_test, mlp_pred, labels=MATERIAL_ORDER)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=MATERIAL_ORDER,
    yticklabels=MATERIAL_ORDER,
)

plt.xlabel("Predicted material")
plt.ylabel("True material")
plt.title("Neural network confusion matrix")

plt.tight_layout()
plt.savefig("../results/neural_network_confusion_matrix.png", dpi=300)
plt.show()

In [ ]:
metrics = pd.DataFrame(
    [
        {"model": "K-Means", "metric": "silhouette_score", "value": silhouette},
        {"model": "KNN", "metric": "accuracy", "value": knn_accuracy},
        {"model": "Neural network", "metric": "accuracy", "value": mlp_accuracy},
    ]
)

metrics.to_csv(RESULTS_DIR / "model_metrics.csv", index=False)

metrics

In [ ]:
## Results

- K-Means was used to explore natural groupings in the spectral feature space.
- KNN and a fully connected neural network were trained to classify films by material.
- Model performance was evaluated using accuracy, classification reports, and confusion matrices.

The thesis pipeline achieved 89.41% recognition accuracy.  
Results in this reconstructed notebook may differ because the original final code and full experimental dataset were not preserved.